# Day 21: Azure Deployment and Continuous Delivery
**Azure hosting, CD pipelines with environments, Key Vault and managed identity, and safe releases**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup
Azure steps run in class only, during the subscription window, in **Azure Cloud Shell** or a VS Code terminal with the Azure CLI. Every other cell runs offline here, including simulators for the pipeline, Key Vault and rollback.

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q fastapi httpx pyyaml

# ===== Option B: VS Code / local =====
# pip install fastapi uvicorn httpx pyyaml
# For live Azure work also: pip install azure-identity azure-keyvault-secrets, and install the Azure CLI.
import os, shutil
shutil.rmtree("deploy-demo", ignore_errors=True)
os.makedirs("deploy-demo/app", exist_ok=True)
os.chdir("deploy-demo")
print("Working in", os.getcwd())

## 2. Choosing a host

In [ ]:
import pandas as pd
workloads = {
    "FastAPI API": {"container": True, "scale_to_zero": False, "full_os": False, "many_services": False},
    "nightly batch job": {"container": True, "scale_to_zero": True, "full_os": False, "many_services": False},
    "legacy Windows app": {"container": False, "scale_to_zero": False, "full_os": True, "many_services": False},
    "40-service platform": {"container": True, "scale_to_zero": False, "full_os": False, "many_services": True},
}

def choose_host(w):
    if w["full_os"]:
        return "Virtual Machine"
    if w["many_services"]:
        return "AKS (with a platform team)"
    if w["scale_to_zero"]:
        return "Container Apps"
    return "App Service or Container Apps"

print(pd.DataFrame([{"workload": k, "host": choose_host(v)} for k, v in workloads.items()]).to_string(index=False))

## 3. A cloud-ready API with a smoke-testable /health

In [ ]:
%%writefile app/main.py
import logging, os, sys
from fastapi import FastAPI

logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
APP_ENV = os.getenv("APP_ENV", "local")
APP_VERSION = os.getenv("APP_VERSION", "dev")
DATABASE_URL = os.getenv("DATABASE_URL", "sqlite:///local.db")   # in Azure: a Key Vault reference

app = FastAPI(title="Tasks API")

@app.get("/health")
def health():
    return {"status": "ok", "env": APP_ENV, "version": APP_VERSION, "db_configured": bool(DATABASE_URL)}

In [ ]:
import importlib, sys
from fastapi.testclient import TestClient
sys.path.insert(0, os.getcwd())
import app.main as m
importlib.reload(m)
print(TestClient(m.app).get("/health").json())

## 4. Deploy with the Azure CLI (in class, Cloud Shell)
Names must be globally unique, so add your initials and a number. **Delete the resource group when you finish.**

In [ ]:
# ===== Option A: Azure Cloud Shell (Bash) =====
# RG=rg-day21-yourname; LOC=centralindia; ACR=acrday21yourname
# az group create -n $RG -l $LOC --tags owner=yourname project=training
# az acr create -g $RG -n $ACR --sku Basic
# az acr build -r $ACR -t tasks-api:1.0 .
#
# --- Option 1: Azure Container Apps (simplest) ---
# az containerapp up --name tasks-api --resource-group $RG --location $LOC \
#   --source . --ingress external --target-port 8000 --env-vars APP_ENV=dev APP_VERSION=1.0
#
# --- Option 2: Azure App Service for containers ---
# az appservice plan create -g $RG -n plan-day21 --is-linux --sku B1
# az webapp create -g $RG -p plan-day21 -n tasks-api-yourname --container-image-name $ACR.azurecr.io/tasks-api:1.0
# az webapp config appsettings set -g $RG -n tasks-api-yourname --settings WEBSITES_PORT=8000 APP_ENV=dev
#
# --- Clean up ---
# az group delete -n $RG --yes --no-wait
print("Azure commands are listed above. Run them in class only.")

## 5. The CD pipeline

In [ ]:
%%writefile azure-pipelines.yml
trigger:
  - main

variables:
  acrServiceConnection: sc-acr          # Docker registry service connection
  azureSubscription: sc-azure            # Azure Resource Manager service connection (workload identity federation)
  repository: tasks-api
  acrLoginServer: acrday21yourname.azurecr.io
  tag: $(Build.BuildId)

stages:
  - stage: Test
    jobs:
      - job: unit
        pool: { vmImage: ubuntu-latest }
        steps:
          - task: UsePythonVersion@0
            inputs: { versionSpec: "3.12" }
          - script: pip install -r requirements.txt pytest httpx && python -m pytest -q
            displayName: Tests

  - stage: Build
    dependsOn: Test
    jobs:
      - job: image
        pool: { vmImage: ubuntu-latest }
        steps:
          - task: Docker@2
            displayName: Build and push image
            inputs:
              command: buildAndPush
              containerRegistry: $(acrServiceConnection)
              repository: $(repository)
              tags: $(tag)

  - stage: DeployDev
    dependsOn: Build
    jobs:
      - deployment: dev
        environment: dev
        pool: { vmImage: ubuntu-latest }
        strategy:
          runOnce:
            deploy:
              steps:
                - task: AzureContainerApps@1
                  inputs:
                    azureSubscription: $(azureSubscription)
                    containerAppName: tasks-api-dev
                    resourceGroup: rg-day21-dev
                    imageToDeploy: $(acrLoginServer)/$(repository):$(tag)
                - script: curl --fail --retry 5 --retry-delay 10 https://$(devUrl)/health
                  displayName: Smoke test dev

  - stage: DeployProd
    dependsOn: DeployDev
    jobs:
      - deployment: prod
        environment: production          # add an approval check on this environment in the UI
        pool: { vmImage: ubuntu-latest }
        strategy:
          runOnce:
            deploy:
              steps:
                - task: AzureContainerApps@1
                  inputs:
                    azureSubscription: $(azureSubscription)
                    containerAppName: tasks-api
                    resourceGroup: rg-day21-prod
                    imageToDeploy: $(acrLoginServer)/$(repository):$(tag)
                - script: curl --fail --retry 5 --retry-delay 10 https://$(prodUrl)/health
                  displayName: Smoke test prod

In [ ]:
import yaml
p = yaml.safe_load(open("azure-pipelines.yml"))
for st in p["stages"]:
    job = st["jobs"][0]
    env = job.get("environment", "")
    print(f"{st['stage']:<11} after {st.get('dependsOn', '-'):<10} {'environment=' + env if env else ''}")

## 6. Variables vs secrets
Secret variables are masked in logs. Masking is a safety net: never print a secret.

In [ ]:
settings = {"region": "centralindia", "appName": "tasks-api", "imageTag": "101", "logLevel": "INFO",
            "dbPassword": "S3cret-db-pass", "jwtSecret": "jwt-signing-value", "apiKey": "training-api-key-value"}
SECRET_WORDS = ("password", "secret", "key", "token", "connection")
secrets = {k: v for k, v in settings.items() if any(w in k.lower() for w in SECRET_WORDS)}
plain = {k: v for k, v in settings.items() if k not in secrets}
print("plain :", list(plain))
print("secret:", list(secrets), "-> Key Vault")

def mask(line):
    for v in secrets.values():
        line = line.replace(v, "***")
    return line

print(mask(f"connecting to db with password={settings['dbPassword']}"))

## 7. NEW (v4): Key Vault and managed identity
A small **offline simulator** of the real flow: the app has an identity, the identity has a role on the vault, the app gets a short-lived token and reads the secret. The real SDK calls are in the next cell.

In [ ]:
import time, uuid

class FakeAzure:
    """Offline stand-in for Microsoft Entra ID + Key Vault, for practice only."""
    def __init__(self):
        self.identities, self.roles, self.tokens = {}, {}, {}
        self.vault = {"db-url": ["postgresql://smartdesk@db-dev:5432/app"]}   # versions, newest last
        self.audit = []

    def assign_identity(self, app):
        self.identities[app] = str(uuid.uuid4())
        return self.identities[app]

    def grant(self, app, role):
        self.roles.setdefault(app, set()).add(role)

    def get_token(self, app, lifetime_s=3600):
        if app not in self.identities:
            raise PermissionError("no managed identity")
        tok = uuid.uuid4().hex
        self.tokens[tok] = (app, time.time() + lifetime_s)
        return tok

    def get_secret(self, token, name):
        app, expires = self.tokens.get(token, (None, 0))
        ok = app and time.time() < expires and "Key Vault Secrets User" in self.roles.get(app, set())
        self.audit.append({"app": app, "secret": name, "result": "OK" if ok else "403"})
        if not ok:
            raise PermissionError("403 Forbidden")
        return self.vault[name][-1]

    def rotate(self, name, value):
        self.vault[name].append(value)

az = FakeAzure()
az.assign_identity("tasks-api")
az.grant("tasks-api", "Key Vault Secrets User")
az.assign_identity("report-job")               # identity, but no role

for app_name in ["tasks-api", "report-job"]:
    try:
        v = az.get_secret(az.get_token(app_name), "db-url")
        print(f"{app_name:<10} -> OK, version {len(az.vault['db-url'])}")
    except PermissionError as e:
        print(f"{app_name:<10} -> {e}")

az.rotate("db-url", "postgresql://smartdesk@db-dev-2:5432/app")
az.get_secret(az.get_token("tasks-api"), "db-url")
print("after rotation -> version", len(az.vault["db-url"]), "read without a code change")
print(pd.DataFrame(az.audit))

In [ ]:
# ===== Real Azure (in class): the same flow =====
# az webapp identity assign -g $RG -n tasks-api-yourname                  # or: az containerapp identity assign --system-assigned
# az role assignment create --assignee <principalId> --role "Key Vault Secrets User" --scope <key-vault-resource-id>
# az webapp config appsettings set -g $RG -n tasks-api-yourname --settings \
#   "DATABASE_URL=@Microsoft.KeyVault(SecretUri=https://<vault>.vault.azure.net/secrets/db-url/)"
#
# Python SDK (pip install azure-identity azure-keyvault-secrets):
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
client = SecretClient("https://<vault>.vault.azure.net", DefaultAzureCredential())
print(client.get_secret("db-url").value[:12], "...")

### Key Vault in the pipeline
The `AzureKeyVault@2` task fetches only the secrets a stage needs. They arrive as secret variables, masked in logs.

In [ ]:
kv_step = yaml.safe_load("""
- task: AzureKeyVault@2
  inputs:
    azureSubscription: sc-azure
    KeyVaultName: kv-smartdesk-dev
    SecretsFilter: 'db-url,jwt-secret'
    RunAsPreJob: false
- script: python migrate.py
  env:
    DATABASE_URL: $(db-url)          # map only what this step needs
""")
print("Secrets fetched:", kv_step[0]["inputs"]["SecretsFilter"])
print("Mapped into env:", list(kv_step[1]["env"]))

## 8. End to end: a release simulator
The same tag moves dev, then (after approval) prod. A failed smoke test stops promotion. Rollback redeploys the previous good tag.

In [ ]:
class Release:
    def __init__(self):
        self.history = {"dev": [], "prod": []}

    def smoke(self, tag, healthy):
        return healthy.get(tag, True)

    def deploy(self, env, tag, healthy):
        self.history[env].append(tag)
        if self.smoke(tag, healthy):
            return True
        bad = self.history[env].pop()
        previous = self.history[env][-1] if self.history[env] else None
        print(f"   {env}: smoke test FAIL for {bad}" + (f" -> rolled back to {previous}" if previous else ""))
        return False

    def promote(self, tag, approved=True, healthy=None, healthy_prod=None):
        healthy = healthy or {}
        if not self.deploy("dev", tag, healthy):
            return f"{tag}: stopped in dev"
        if not approved:
            return f"{tag}: dev OK, waiting for approval"
        if not self.deploy("prod", tag, healthy_prod or healthy):
            return f"{tag}: prod failed, rolled back"
        return f"{tag}: dev OK -> approved -> prod OK"

r = Release()
print(r.promote("101"))
print(r.promote("102", healthy={"102": False}))
print(r.promote("103", healthy_prod={"103": False}))
print("Running now:", {env: (h[-1] if h else None) for env, h in r.history.items()})

### TODO 1 (SmartDesk step 18): write a one-page runbook for SmartDesk: deploy, verify, read logs, roll back, rotate the db-url secret, clean up. Paste it in a Markdown cell below.

In [ ]:
# TODO: write your code here

## 9. Check yourself
1. Why was report-job refused even though it had an identity?
2. Which tag is running in prod at the end, and why?
3. Which settings above belong in Key Vault?

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)